<a href="https://colab.research.google.com/github/pascal-maker/agents/blob/main/Jurimeshpascalipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q gdown langchain langchain-experimental sentence-transformers langchain-text-splitters langchain-community streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.2/209.2 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 111.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 84.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 91.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5

# Prelims

In [ ]:
# Assumptions, requirements and choices:
#
# - Input documents won't have predictable structure
#   > Stick to chunking approach that does not assume a particular document structure of sections, paragraphs, etc.
#
# - Exact extraction of text & prefer false positives over false negatives, i.e., over- rather than under-retrieval preferred
#   > No generation & put the exact occurence of answers in extracted text as priority when evaluating approaches
#
# - For this MVP, I chose to only go with pre-trained sentence embedders, to keep it simple (no data engineering and training necessary)
#   and to illustrate problem solving ability under constrained conditions
#
# Approach summary:
#
# I use two filtering steps:
#
# 1. An initial coarse one identifying the most relevant sentences and keeping the context centered around them.
#
# 2. A more fine-grained step to pinpoint the most important sentences among the previous extracted contexts.
#
# Relevancy is computed using a pre-trained sentence embedder and the cosine similarity.
#
#
# Disclaimers:
#
# - ChatGPT used to generate functions for speed
#
# - Certain parameters were set to try out the approach,
#   however, a modest dataset is necessary to set them such that the approach generalizes
#
# =============================================================================
# Installs

!pip install -q gdown langchain langchain-experimental sentence-transformers langchain-text-splitters langchain-community

# =============================================================================
# Data

!gdown --folder "https://drive.google.com/drive/folders/1-idkcV-TYmo7MxT3I5fY9tmePqS7bJKm"

# =============================================================================
# Constants

EXAMPLES_DIR = "/content/examples"

# =============================================================================
# Imports

# from langchain.vectorstores import FAISS
# from langchain_community.vectorstores import FAISS
# import faiss
from itertools import combinations
from langchain.chains import RetrievalQA
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.llms import CTransformers
from langchain.schema import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_experimental.text_splitter import SemanticChunker
from langchain_text_splitters import RecursiveCharacterTextSplitter
from openai import OpenAI
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from typing import List, Tuple, Callable, Dict
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import re
import seaborn as sns

# =============================================================================
# Data exploration

def print_txt_file(directory: str, filename: str, num_lines: int = -1, encoding: str = "utf-8") -> None:
    """
    Reads and prints the contents of a text file.

    Args:
        directory (str): The path to the directory containing the file.
        filename (str): The name of the text file.
        num_lines (int): Number of lines to print. Default (-1) prints the entire file.
        encoding (str): Encoding type (default is "utf-8").

    Raises:
        FileNotFoundError: If the file does not exist in the given directory.
    """
    file_path = os.path.join(directory, filename)

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File '{filename}' not found in directory '{directory}'")

    with open(file_path, "r", encoding=encoding) as file:
        if num_lines == -1:
            print(file.read())  # Print entire file
        else:
            for _ in range(num_lines):
                line = file.readline()
                if not line:
                    break  # Stop if end of file is reached
                print(line, end="")  # Avoid double newlines

print_txt_file(EXAMPLES_DIR, "document-one.txt")
print_txt_file(EXAMPLES_DIR, "document-one-qa.txt")
print_txt_file(EXAMPLES_DIR, "document-two.txt")
print_txt_file(EXAMPLES_DIR, "document-two-qa.txt")
print_txt_file(EXAMPLES_DIR, "document-three.txt")
print_txt_file(EXAMPLES_DIR, "document-three-qa.txt")

# =============================================================================
# NOTE
#
# The answer to the third question `What are the IP transfer provisions` is not extracted from the source document as far as I can see.
#
# From the example documents, I initially conclude the following next steps:
#
# 1. Extract the questions and answers from the `document-**-qa.txt ` files.
#    Questions necessary to test the approach, answers to evaluate it.
#
# 2. Chunk `document-**.txt` files.
#
# 3. For an input question retrieve appropriate chunks.
#
# 4. Check the answer is literally present in the retrieved chunks.

# =============================================================================
# Data engineering

def parse_qa_file(file_path: str) -> list:
    """
    Parses a QA text file and extracts questions, answers, and adds the source document content.

    Args:
        file_path (str): Path to the QA file.

    Returns:
        list: A list of dictionaries, each containing 'filename', 'question', 'answer', and 'document'.
    """
    data = []
    filename = os.path.basename(file_path)

    # Read the content of the corresponding source document (same file name but without '-qa')
    document_path = file_path.replace("-qa", "")
    with open(document_path, "r", encoding="utf-8") as doc_file:
        document_content = doc_file.read()

    # Read the Q&A pairs from the QA file
    with open(file_path, "r", encoding="utf-8") as qa_file:
        content = qa_file.read()

    # Use regex to extract Q&A pairs
    qa_pairs = re.findall(r"Q:\s*(.*?)\nA:\s*(.*?)(?=\nQ:|\Z)", content, re.DOTALL)

    # Append each Q&A pair with the source document content
    for question, answer in qa_pairs:
        data.append({
            "filename": filename,
            "question": question.strip(),
            "answer": answer.strip(),
            "document": document_content.strip()  # Add the document content to each QA pair
        })

    return data

def create_qa_dataframe(directory: str) -> pd.DataFrame:
    """
    Reads all QA text files in a directory and creates a Pandas DataFrame.

    Args:
        directory (str): Path to the directory containing QA files.

    Returns:
        pd.DataFrame: A DataFrame with columns ['filename', 'question', 'answer', 'document'].
    """
    all_data = []

    for file in os.listdir(directory):
        if file.endswith("qa.txt"):
            file_path = os.path.join(directory, file)
            # Parse the Q&A file and retrieve the relevant data
            all_data.extend(parse_qa_file(file_path))

    return pd.DataFrame(all_data, columns=["filename", "question", "answer", "document"])

DF = create_qa_dataframe(EXAMPLES_DIR)
# Incorrect answer I suspect (not found in document)
DF = DF.drop(index=11)
DF.drop(columns=["document"])

# =============================================================================
# Verify the dataset ✅
#
# E.g., `What is the rental price?` from `document-one-qa.txt` corresponds to dataframe:
#
# ```
#  6 Rent and indexation
# 6.1 The base rent is EUR 876 per month. The rent is payable in advance no later than the fifth Working Day of the month to which it relates. The rent must be transferred to account number BE 55 5656 5656 5656 in the name of the Lessor with reference "Rent Berlaar". The rent shall be due for the first time for the month of November, and this no later than November 1, 2023.
#
# 6.2 The rent shall be annually adjusted on the anniversary of this Agreement's date to the cost of living (with the understanding that this will happen for the first time in November 2024) according to the following formula:
#
# new rent = (base rent x new index figure) / initial index figure,
#
# where:
# (i) the base rent is the rent mentioned in Article 6.1
# (ii) the new index figure is the health index figure for January of the year in which the rent is adjusted and
# (iii) the initial index figure is the health index figure for November 2023.
# ```

with open(f"{EXAMPLES_DIR}/document-one.txt") as f:
    TEST_FILE_ONE = f.read()
with open(f"{EXAMPLES_DIR}/document-two.txt") as f:
    TEST_FILE_TWO = f.read()
with open(f"{EXAMPLES_DIR}/document-three.txt") as f:
    TEST_FILE_THREE = f.read()

TEST_QUESTION_ONE = "What is the rental price?"
TEST_QUESTION_TWO = "What is the rental price?"
TEST_QUESTION_THREE = "Which court should be used in case of disputes?"

TEST_ANSWER_ONE = "6 Rent and indexation 6.1 The base rent is EUR 876 per month. The rent is payable in advance no later than the fifth Working Day of the month to which it relates. The rent must be transferred to account number BE 55 5656 5656 5656 in the name of the Lessor with reference \"Rent Berlaar\". The rent shall be due for the first time for the month of November, and this no later than November 1, 2023. 6.2 The rent shall be annually adjusted on the anniversary of this Agreement's date to the cost of living (with the understanding that this will happen for the first time in November 2024) according to the following formula: new rent = (base rent x new index figure) / initial index figure, where: (i) the base rent is the rent mentioned in Article 6.1 (ii) the new index figure is the health index figure for January of the year in which the rent is adjusted and (iii) the initial index figure is the health index figure for November 2023."
TEST_ANSWER_TWO = "Article 6. Ground rent and default interest The emphyteutic fee amounts to 15,000.00 EUR per year and is periodically payable in twelve monthly installments of ... 1,250.00 EUR per month. Payment must be made before the start of each month to account number ... 789789789. The first payment of the ground rent is due from the execution of the notarial deed. When the emphyteutic fee is not paid on time by the emphyteutic lessee, it shall, from the date of enforceability and without requiring an explicit notice of default, legally incur default interest of 10% per year from the due date until full payment, without prejudice to the other rights of the city."
TEST_ANSWER_THREE = "The courts of Ghent shall have exclusive jurisdiction to settle any dispute arising out of or in connection with this Agreement (including a dispute relating to non-contractual obligations arising out of or in connection with this Agreement) which the Parties are unable to settle amicably."

# =============================================================================
# Technique explorations
#
# Possible approaches to splitting or chunking documents:
#
# - Constant chunking
# - Basic sentence splitting
# - SpaCy based sentence splitting
# - Stanford stanza sentence splitting
# - Recursive chunking
# - Semantic chunking
# - Hierarchical chunks: paragraph- to sentence-based
# - openAI chunking
# - Chunk file into a hierarchy, e.g., preamble, body, conclusion, with each having subchunks, etc.
#
# =============================================================================
# Helper functions
#
# Text splitting & chunking

def sentence_chunking_basic(text):
    """
    Split the text into sentences based on common sentence-ending punctuation marks (., ?, !).
    Handles simple sentence boundaries and avoids overly aggressive merging of sentences.
    """
    # Regular expression to split the text at sentence boundaries
    sentences = re.split(r'(?<!\.\.\.)(?<=\.|\?|\!)(?=\s)', text.strip())

    # Clean up any empty sentences that might have been created
    sentences = [sentence.strip() for sentence in sentences if sentence.strip()]

    return sentences

!pip install -q spacy
!python -m spacy download en_core_web_sm

def sentence_chunking_spacy(text):
    import spacy
    # Load the English model
    nlp = spacy.load("en_core_web_sm")
    doc = nlp(text)
    return [sent.text.strip() for sent in doc.sents]

# Uncomment if needed:
# !pip install stanza
# !python -m stanza.download('en')
# def sentence_chunking_stanza(text):
#     import stanza
#     # Download and load the English model
#     stanza.download('en')
#     nlp = stanza.Pipeline('en')
#     doc = nlp(text)
#     return [sentence.text.strip() for sentence in doc.sentences]

def constant_chunk(text: str, chunk_size: int = 50) -> List[str]:
    """
    Splits text into chunks of specified size.
    """
    chunks = [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]
    return chunks

def semantic_chunk(
        text: str,
        breakpoint_threshold_type: str,
        breakpoint_threshold_amount: float,
        model_name="sentence-transformers/all-MiniLM-L6-v2",
    ) -> List[str]:
    """
    Splits text into chunks using SemanticChunker, with a specified chunk size.
    Uses a HuggingFace model to embed the text and split it semantically based on the threshold.
    """
    # Initialize the HuggingFace Embeddings model
    hf_embeddings = HuggingFaceEmbeddings(model_name=model_name)

    # Initialize the SemanticChunker
    text_splitter = SemanticChunker(
        hf_embeddings,
        breakpoint_threshold_type=breakpoint_threshold_type,
        breakpoint_threshold_amount=breakpoint_threshold_amount,
    )

    # Use the chunker to split the text semantically
    chunks = text_splitter.split_text(text)

    return chunks

def recursive_chunk(
    text: str,
    chunk_size: int = 1000,
    chunk_overlap: int = 200,
    separators=["\n\n", "\n", " "],
    ) -> List[str]:
    """
    Splits text into chunks using RecursiveCharacterTextSplitter.
    Returns a list of strings.
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        is_separator_regex=False,
        separators=separators,
    )
    chunks = text_splitter.create_documents([text])

    return [chunk.page_content for chunk in chunks]  # Extract the raw text

def chunk_text_with_openai(text: str) -> list:
    """
    Chop up the input text into smaller chunks, send them to OpenAI for processing,
    and combine the results.
    """
    chunk_size = 2000  # Define your desired chunk size
    chunks = []

    # Split the input text into smaller chunks of specified size
    for i in range(0, len(text), chunk_size):
        chunks.append(text[i:i + chunk_size])

    combined_result = []

    # Process each chunk with OpenAI API
    for chunk in chunks:
        # Prepare the prompt for OpenAI's model
        prompt = f"""
        Chunk the following legal text appropriately, keep relevant elements together.
        Include a separator (---) between chunks.
        Ensure the output reflects the original wording as much as possible.

        {chunk}
        """

        # Send the request to OpenAI's API (assumes client is already defined and initialized)
        completion = client.chat.completions.create(
            model="gpt-4",
            messages=[
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": prompt}
            ],
            max_tokens=2000,  # Adjust if needed based on the chunk size
        )

        # Extract the response content and append it to combined results
        combined_result.append(completion.choices[0].message.content)

    # Combine all processed chunks with separator "---" between them
    return '---'.join(combined_result)

def add_indices_to_chunks(chunks: List[str]) -> List[Tuple[int, str]]:
    """
    Adds index numbers to each chunk.

    Returns:
        List of tuples (chunk_index, chunk_text).
    """
    return [(i, chunk) for i, chunk in enumerate(chunks)]

# =============================================================================
# Embedding

def embed_text(text: str, model: SentenceTransformer) -> np.ndarray:
    return model.encode(text)

def embed_chunks(chunks: List[str], model: SentenceTransformer) -> List[np.ndarray]:
    return [embed_text(chunk, model) for chunk in chunks]

MODEL = SentenceTransformer('all-MiniLM-L6-v2')

# =============================================================================
# Retrieval

def top_k_chunks(similarities: List[float], k: int) -> List[int]:
    """
    Selects the top-k most similar chunks based on cosine similarity.
    """
    sorted_indices = np.argsort(similarities)[::-1]  # Sort in descending order
    return sorted_indices[:k]

def above_threshold_chunks(similarities: List[float], threshold: float) -> List[int]:
    """
    Selects chunks with similarity above the threshold.
    """
    return [i for i, sim in enumerate(similarities) if sim >= threshold]

def percentile_based_chunks(
        similarities: List[float],
        percentile=75
    ) -> List[int]:
    threshold = np.percentile(similarities, percentile)
    return above_threshold_chunks(similarities, threshold)

def get_percentile_threshold(similarities, percentile=75):
    return np.percentile(similarities, percentile)

# =============================================================================
# Plotting

def plot_similarity_cdf(df, col):
    sorted_similarities = np.sort(df[col])

    # Compute CDF values
    cdf_values = np.arange(1, len(sorted_similarities) + 1) / len(sorted_similarities)

    # Plot CDF
    plt.figure(figsize=(10, 5))
    sns.lineplot(x=sorted_similarities, y=cdf_values, marker="o", linestyle="-", color="blue")
    plt.xlabel("Cosine Similarity Score")
    plt.ylabel("Cumulative Probability")
    plt.title("Cumulative Distribution Function (CDF) of Similarity Scores")
    plt.grid(True)
    plt.show()

def plot_similarity(df, col):
    # Plot CDF
    plt.figure(figsize=(10, 5))
    sns.lineplot(x=df.index, y=df[col], marker="o", linestyle="-", color="blue")
    plt.xlabel("Chunk")
    plt.ylabel("Cosine Similarity Score")
    plt.title("Similarity Scores")
    plt.grid(True)
    plt.show()

# =============================================================================
# Evaluation

def preprocess_text(text: str) -> str:
    """
    Preprocess text by:
    - Lowercasing the text
    - Removing newlines and extra spaces
    - Removing punctuation
    """
    # Convert to lowercase
    text = text.lower()
    # Remove newlines and extra spaces
    text = re.sub(r'\s+', ' ', text)
    # Optionally remove punctuation (if desired)
    text = re.sub(r'[^\w\s]', '', text)
    return text

def score_dataframe(df: pd.DataFrame, model: SentenceTransformer, col="relevant_context") -> pd.DataFrame:
    """
    Calculate average similarity between the answer and the concatenated relevant chunks for each row in the DataFrame.
    Adds 'similarity_score' and 'answer_present' columns to the DataFrame.
    """
    similarity_scores = []
    answer_presence = []

    for index, row in df.iterrows():
        # Concatenate relevant chunks into one string and preprocess
        concatenated_chunks = row[col]
        concatenated_chunks = preprocess_text(concatenated_chunks)

        # Preprocess the answer
        answer = preprocess_text(row['answer'])

        # Embed the answer and concatenated chunks using the same model
        answer_embedding = model.encode(answer).reshape(1, -1)  # Ensure 2D shape
        chunks_embedding = model.encode(concatenated_chunks).reshape(1, -1)  # Ensure 2D shape

        # Compute cosine similarity
        similarity = cosine_similarity(answer_embedding, chunks_embedding)
        similarity_scores.append(similarity)

        # Check if the answer is literally present in the concatenated chunks (after preprocessing)
        answer_present = answer in concatenated_chunks
        answer_presence.append(answer_present)

    # Add new evaluation columns
    df['similarity_score'] = similarity_scores
    df['answer_present'] = answer_presence

    return df

def average_string_length_by_filename(df: pd.DataFrame, col: str) -> pd.DataFrame:
    """
    Groups the DataFrame by 'filename' and calculates the average length of strings
    in the specified column, and computes the percentage of this average length
    relative to the average length in the 'document' column.

    Args:
        df (pd.DataFrame): The input DataFrame containing 'filename', a column (`col`), and 'document' columns.
        col (str): The name of the column for which to calculate the average string length.

    Returns:
        pd.DataFrame: A DataFrame with 'filename', 'avg_length' for the specified column,
                       the average length of 'document' column, and the computed 'percentage'.
    """
    # Calculate the average string length of the specified column
    avg_lengths = df.groupby('filename')[col].apply(lambda x: x.str.len().mean()).reset_index()

    # Calculate the average string length of the 'document' column
    total_lengths = df.groupby('filename')['document'].apply(lambda x: x.str.len().mean()).reset_index()

    # Rename columns for clarity
    avg_lengths = avg_lengths.rename(columns={col: 'avg_length'})
    total_lengths = total_lengths.rename(columns={'document': 'document_avg_length'})

    # Merge the two DataFrames on 'filename'
    percentage = pd.merge(avg_lengths, total_lengths, on='filename')

    # Calculate the percentage of avg_length over document_avg_length
    percentage['percentage'] = (percentage['avg_length'] / percentage['document_avg_length']) * 100

    return percentage

def summary_statistics(df: pd.DataFrame, col="relevant_context") -> pd.DataFrame:
    """
    Calculate summary statistics for the DataFrame.
    """
    similarity_scores = df['similarity_score'].tolist()
    answer_presence = df['answer_present'].tolist()

    print(f"Avg. similarity score: {np.mean(similarity_scores)}")
    print(f"Avg. answer presence: {np.mean(answer_presence)}")
    print(f"Avg. string length: {average_string_length_by_filename(df, col)['percentage'].mean()}")

# =============================================================================
# Baselines: chunk & retrieve
#
# Ultimately sentence chunking was chosen,
# because the others were not as reliable.
# E.g., semantic chunking did not appropriately capture the different articles in separate chunks,
# at least without preprocessing, making it rather unhelpful.

def apply_to_qa_df(
    df,
    chunking_func,
    chunking_params,
    retrieval_func,
    retrieval_params,
    model,
    top_k=3,
    doc_col="document",
    relevant_context_col="relevant_context",
):
    df = df.copy()
    relevant_contexts = []  # Ensure the list is defined

    for _, row in df.iterrows():
        question = row['question']
        document = row[doc_col]

        # Step 1: Chunking
        chunks = chunking_func(document, **chunking_params)

        # Step 2: Embed chunks
        embedded_chunks = np.array(embed_chunks(chunks, model))

        # Step 3: Reshape embeddings to the correct shape (1, -1)
        embedded_chunks_reshaped = [
            chunk_embedding.reshape(1, -1) for chunk_embedding in embedded_chunks
        ]

        # Step 4: Embed the question
        question_embedding = embed_chunks([question], model)[0].reshape(1, -1)

        # Step 5: Calculate cosine similarity between the question and each chunk
        similarities = [
            cosine_similarity(question_embedding, chunk_embedding)
            for chunk_embedding in embedded_chunks_reshaped
        ]

        # Step 6: Retrieve Top-k chunk indices using the retrieval function
        relevant_chunk_indices = retrieval_func(similarities, **retrieval_params)

        # Convert indices to integers (if they are NumPy scalars/arrays)
        relevant_chunk_indices = [
            idx.item() if isinstance(idx, np.ndarray) else int(idx)
            for idx in relevant_chunk_indices
        ]

        # Select and sort relevant chunks
        relevant_chunks = [(i, chunks[i]) for i in relevant_chunk_indices]
        relevant_chunks = sorted(relevant_chunks, key=lambda x: x[0])
        relevant_chunks = [chunk for _, chunk in relevant_chunks]

        # Step 7: Concatenate the relevant chunks into a single context string
        relevant_contexts.append(" ".join(relevant_chunks))

    # Add the relevant context as a new column in the DataFrame
    df[relevant_context_col] = relevant_contexts

    return df

# =============================================================================
# Example of using the conversion logic outside the function (if needed):
# relevant_chunk_indices = retrieval_func(similarities, **retrieval_params)
# relevant_chunk_indices = [
#     idx.item() if isinstance(idx, np.ndarray) else int(idx)
#     for idx in relevant_chunk_indices
# ]
# relevant_chunks = [(i, chunks[i]) for i in relevant_chunk_indices]

# =============================================================================
# Testing sentence chunking basic

for chunk in sentence_chunking_basic(
        TEST_FILE_ONE,
    ):
    print(chunk)
    print("---")

# =============================================================================
# Second definition of apply_to_qa_df with explicit index conversion

def apply_to_qa_df(
    df,
    chunking_func,
    chunking_params,
    retrieval_func,
    retrieval_params,
    model,
    top_k=3,
    doc_col="document",
    relevant_context_col="relevant_context",
):
    df = df.copy()
    relevant_contexts = []  # 🔥 FIX: Ensure the list is defined

    for _, row in df.iterrows():
        question = row['question']
        document = row[doc_col]

        # Stap 1: Chunking
        chunks = chunking_func(document, **chunking_params)

        # Stap 2: Embed chunks
        embedded_chunks = np.array(embed_chunks(chunks, model))

        # Stap 3: Reshape embeddings naar juiste vorm (1, -1)
        embedded_chunks_reshaped = [
            chunk_embedding.reshape(1, -1) for chunk_embedding in embedded_chunks
        ]

        # Stap 4: Embed de vraag
        question_embedding = embed_chunks([question], model)[0].reshape(1, -1)

        # Stap 4: Cosine Similarity berekenen
        similarities = [
            cosine_similarity(question_embedding, chunk_embedding)
            for chunk_embedding in embedded_chunks_reshaped
        ]

        # Stap 5: Top-k chunks ophalen
        relevant_chunk_indices = retrieval_func(similarities, **retrieval_params)

        # 🔥 Oplossing voor indices:
        relevant_chunk_indices = [
            idx.item() if isinstance(idx, np.ndarray) else int(idx)
            for idx in relevant_chunk_indices
        ]

        # Relevante chunks selecteren en sorteren
        relevant_chunks = [(i, chunks[i]) for i in relevant_chunk_indices]
        relevant_chunks = sorted(relevant_chunks, key=lambda x: x[0])
        relevant_chunks = [chunk for _, chunk in relevant_chunks]

        # Stap 6: Relevante context samenstellen
        relevant_contexts.append(" ".join(relevant_chunks))

    # Nieuwe kolom toevoegen aan DataFrame
    df[relevant_context_col] = relevant_contexts

    return df

# =============================================================================
# Contextual sentence chunking with sliding window similarity and threshold filtering

def rolling_window_similarity(question, document, sentence_chunking_func, model_name="all-MiniLM-L6-v2", window_size=2, stride=1):
    """
    Compute rolling window similarity and track the maximum similarity score for each sentence.

    Parameters:
        question (str): The input question.
        document (str): The document as a single string.
        sentence_chunking_func (func): Function to split document into sentences.
        model_name (str): Sentence-transformer model.
        window_size (int): Number of sentences per rolling window.
        stride (int): Step size for the rolling window.

    Returns:
        pandas.DataFrame: Sentences and their maximum similarity scores.
    """
    model = SentenceTransformer(model_name)

    # Split document into sentences
    sentences = sentence_chunking_func(document)
    if not sentences:
        raise ValueError("No sentences extracted from document.")

    sentence_max_similarity = {sentence: 0.0 for sentence in sentences}

    # Generate rolling windows (ensuring coverage of entire document)
    windows = []
    num_sentences = len(sentences)

    for i in range(0, num_sentences, stride):
        window = sentences[i : i + window_size]
        if len(window) == window_size:
            windows.append(window)
        else:
            # Include the last smaller window at the end of document
            if i + window_size > num_sentences:
                windows.append(sentences[-window_size:])
                break

    window_texts = [" ".join(window) for window in windows]

    if not window_texts:
        raise ValueError("No window texts generated from the document.")

    # Compute embeddings
    question_embedding = model.encode(question, convert_to_tensor=True)
    window_embeddings = model.encode(window_texts, convert_to_tensor=True)

    # Compute cosine similarities
    similarities = cosine_similarity(
        question_embedding.cpu().reshape(1, -1),
        window_embeddings.cpu()
    )[0]

    # Update max similarity per sentence
    for window, similarity in zip(windows, similarities):
        for sentence in window:
            sentence_max_similarity[sentence] = max(sentence_max_similarity[sentence], similarity)

    # Create DataFrame
    df = pd.DataFrame({
        'Sentence': sentences,
        'Max_Similarity': [sentence_max_similarity[sentence] for sentence in sentences]
    })

    return df

def filter_and_concatenate(df, threshold):
    """
    Filters sentences with Max_Similarity above the threshold and concatenates them into a single text.

    Parameters:
        df (pandas.DataFrame): DataFrame with 'Sentence' and 'Max_Similarity' columns.
        threshold (float): Minimum similarity score for a sentence to be included.

    Returns:
        str: Concatenated text of relevant sentences.
    """
    filtered_df = df[df['Max_Similarity'] >= threshold]
    return " ".join(filtered_df['Sentence'].tolist())

# =============================================================================
# Additional rolling window similarity (alternative version)

def rolling_window_similarity_alt(question, document, sentence_chunking_func, model_name="all-MiniLM-L6-v2", window_size=2, stride=1):
    """
    Compute rolling window similarity and track the maximum similarity score for each sentence.

    Parameters:
        question (str): The input question.
        document (str): The document as a single string.
        sentence_chunking_func (func): Function to split document into sentences.
        model_name (str): Sentence-transformer model.
        window_size (int): Number of sentences per rolling window.
        stride (int): Step size for the rolling window.

    Returns:
        pandas.DataFrame: Sentences and their maximum similarity scores.
    """
    model = SentenceTransformer(model_name)

    # Split document into sentences
    sentences = sentence_chunking_func(document)
    if not sentences:
        raise ValueError("No sentences found in the document after chunking.")

    sentence_max_similarity = {sentence: 0.0 for sentence in sentences}

    # Generate rolling windows
    windows = []
    half_window = window_size // 2
    for i in range(half_window, len(sentences) - half_window, stride):
        window = sentences[i - half_window : i + half_window + 1]
        windows.append(window)

    window_texts = [" ".join(window) for window in windows]

    if not window_texts:
        raise ValueError("No window texts generated from the document.")

    # Compute embeddings
    question_embedding = model.encode([question]).reshape(1, -1)
    window_embeddings = model.encode(window_texts).reshape(1, -1)

    if question_embedding.ndim != 2 or window_embeddings.ndim != 2:
        raise ValueError("Expected 2D array for embeddings.")

    # Compute cosine similarity
    similarities = cosine_similarity(question_embedding, window_embeddings)[0]

    # Update max similarity for each sentence
    for i, (window, similarity) in enumerate(zip(windows, similarities)):
        for sentence in window:
            sentence_max_similarity[sentence] = max(sentence_max_similarity[sentence], similarity)

    # Create DataFrame with sentence-level max similarity
    df = pd.DataFrame({
        'Sentence': sentences,
        'Max_Similarity': [sentence_max_similarity[sentence] for sentence in sentences]
    })

    return df

# =============================================================================
# Testing rolling_window_similarity

TEST_ONE_DF = rolling_window_similarity(
    TEST_QUESTION_ONE,
    TEST_FILE_ONE,
    sentence_chunking_func=sentence_chunking_spacy,
    window_size=10,
    stride=1,
)

# Force correct numeric dtype
TEST_ONE_DF['Max_Similarity'] = pd.to_numeric(TEST_ONE_DF['Max_Similarity'], errors='coerce')
TEST_ONE_DF.dropna(subset=['Max_Similarity'], inplace=True)

threshold = get_percentile_threshold(TEST_ONE_DF['Max_Similarity'], percentile=85)
print(TEST_ONE_DF[TEST_ONE_DF['Max_Similarity'] >= threshold])
print(filter_and_concatenate(TEST_ONE_DF, threshold))

# =============================================================================
# Implementation for updating similarities with window and retrieval

def update_similarities_with_window(similarities: List[float], window_size: int) -> List[float]:
    """
    Updates the similarities by assigning the max similarity within a window centered around each similarity.

    Args:
        similarities (List[float]): List of similarity values.
        window_size (int): The size of the window around each similarity to calculate the max similarity.

    Returns:
        List[float]: The updated similarity values with the max similarity within the window.
    """
    updated_similarities = []
    n = len(similarities)

    for i in range(n):
        # Define the window range (making sure it's within the bounds of the list)
        start = max(0, i - window_size // 2)
        end = min(n, i + window_size // 2 + 1)

        # Extract the similarities within the window
        window = similarities[start:end]

        # Assign the max similarity from the window
        max_similarity = max(window)

        # Append the max similarity to the updated list
        updated_similarities.append(max_similarity)

    return updated_similarities

def window_retrieval(similarities: List[float], retrieval_func: Callable, retrieval_params: Dict, window_size: int) -> List[int]:
    updated_similarities = update_similarities_with_window(similarities, window_size)
    indices = retrieval_func(updated_similarities, **retrieval_params)
    return indices

# =============================================================================
# Experiments

def run_experiments(df: pd.DataFrame, model, window_sizes: list, percentiles: list) -> pd.DataFrame:
    results = []
    for window_size in window_sizes:
        for percentile in percentiles:
            print(f"Running experiment for window_size={window_size} and percentile={percentile}...")

            rec_w_df = apply_to_qa_df(
                df,
                sentence_chunking_spacy,
                {},
                window_retrieval,
                {"window_size": window_size, "retrieval_func": percentile_based_chunks, "retrieval_params": {"percentile": percentile}},
                model,
            )

            rec_w_score_df = score_dataframe(rec_w_df, model)

            # Calculate average similarity and answer presence
            avg_similarity = float(np.mean(rec_w_score_df['similarity_score']))
            avg_answer_presence = float(np.mean(rec_w_score_df['answer_present']))

            # Calculate percentage of avg_length over document_avg_length
            percentage_df = average_string_length_by_filename(rec_w_score_df, 'relevant_context')
            avg_percentage = float(percentage_df['percentage'].mean()) if 'percentage' in percentage_df.columns else np.nan

            print(f"  Avg. Similarity: {avg_similarity:.4f}")
            print(f"  Avg. Answer Presence: {avg_answer_presence:.4f}")
            print(f"  Avg. Percentage: {avg_percentage:.4f}")
            print("-" * 50)

            results.append({
                'window_size': window_size,
                'percentile': percentile,
                'avg_similarity': avg_similarity,
                'avg_answer_presence': avg_answer_presence,
                'avg_percentage': avg_percentage,
            })

    results_df = pd.DataFrame(results)
    return results_df

experiment_results = run_experiments(
    DF,
    MODEL,
    [20, 15, 10, 5, 3, 1],
    [95, 90, 85, 80, 75, 70, 65, 60, 55, 50, 45],
)

experiment_results.to_csv('experiment_results.csv', index=False)

# =============================================================================
# Nth percentile relevant with context window

def relevant_context(question: str, text: str, model: SentenceTransformer) -> pd.DataFrame:
    # Step 1: Chunk the text into sentences
    chunks = sentence_chunking_spacy(text)

    # Step 2: Embed the question and chunks using the SentenceTransformer model
    question_embedding = embed_text(question, model)
    chunk_embeddings = embed_chunks(chunks, model)

    # Step 3: Calculate cosine similarity between the question and each chunk
    similarities = [cosine_similarity([question_embedding], [chunk_embedding])[0][0] for chunk_embedding in chunk_embeddings]

    # Step 4: Create a DataFrame with sentences and similarity scores
    similarity_df = pd.DataFrame({
        'Sentence': chunks,
        'Similarity': similarities
    })

    return similarity_df

TMP_DF = relevant_context(TEST_QUESTION_ONE, TEST_FILE_ONE, MODEL)

def plot_similarity(df, col):
    plt.figure(figsize=(12, 6))
    plt.plot(df.index, df[col], marker='o', linestyle='-', label='Similarity')
    plt.xlabel('Chunk Index')
    plt.ylabel('Similarity Score')
    plt.title('Similarity Scores')
    plt.legend()
    plt.grid(True)
    plt.show()

plot_similarity(TMP_DF, "Similarity")

def indicate_top_percentile(df: pd.DataFrame, percentile: float) -> pd.DataFrame:
    """
    Adds a new column 'Top_Percentile' indicating whether each sentence is in the top nth percentile.

    Parameters:
    df (pd.DataFrame): DataFrame containing 'Sentence' and 'Similarity' columns.
    percentile (float): The percentile threshold (e.g., 90 for top 10% of similarities).

    Returns:
    pd.DataFrame: Updated DataFrame with 'Top_Percentile' column (True/False).
    """
    df = df.copy()
    threshold = df['Similarity'].quantile(percentile / 100)
    df['Top_Percentile'] = df['Similarity'] >= threshold
    return df

def plot_top_percentile_similarity(df: pd.DataFrame):
    plt.figure(figsize=(12, 6))
    plt.plot(df.index, df['Similarity'], marker='o', linestyle='-', label='Similarity')
    top_indices = df[df['Top_Percentile']].index
    plt.scatter(top_indices, df.loc[top_indices, 'Similarity'], color='red', label='Top Percentile', zorder=3)
    plt.xlabel('Chunk Index')
    plt.ylabel('Similarity Score')
    plt.title('Similarity Score per Chunk')
    plt.legend()
    plt.grid(True)
    plt.show()

plot_top_percentile_similarity(indicate_top_percentile(TMP_DF, 95))

def indicate_window_around_top_percentile(df: pd.DataFrame, window_size: int = 3) -> pd.DataFrame:
    df = df.copy()
    df['In_Window'] = False
    top_indices = df[df['Top_Percentile']].index

    for idx in top_indices:
        start = max(0, idx - window_size)
        end = min(len(df) - 1, idx + window_size)
        df.loc[start:end, 'In_Window'] = True

    return df

def plot_window_similarity(df: pd.DataFrame):
    plt.figure(figsize=(12, 6))
    plt.plot(df.index, df['Similarity'], marker='o', linestyle='-', label='Similarity', alpha=0.5)
    window_indices = df[df['In_Window']].index
    plt.scatter(window_indices, df.loc[window_indices, 'Similarity'], color='blue', label='In Window', zorder=3)
    plt.xlabel('Chunk Index')
    plt.ylabel('Similarity Score')
    plt.title('Similarity Score for Chunks in Window')
    plt.legend()
    plt.grid(True)
    plt.show()

plot_window_similarity(indicate_window_around_top_percentile(indicate_top_percentile(TMP_DF, 95), 3))

def concatenate_window_sentences(df: pd.DataFrame) -> str:
    return " ".join(df[df['In_Window']]['Sentence'])

print(concatenate_window_sentences(indicate_window_around_top_percentile(indicate_top_percentile(TMP_DF, 95), 3)))

def apply_nth_percentile_context_window_retrieval_to_df(df: pd.DataFrame, model, n=95, window_size=3) -> pd.DataFrame:
    df = df.copy()
    df['relevant_context'] = df.apply(
        lambda row: concatenate_window_sentences(
            indicate_window_around_top_percentile(
                indicate_top_percentile(
                    relevant_context(row['question'], row['document'], model),
                    n
                ),
                window_size,
            )
        ),
        axis=1
    )
    return df

N_DF = apply_nth_percentile_context_window_retrieval_to_df(DF, MODEL, n=97, window_size=7)
N_DF = score_dataframe(N_DF, MODEL)
summary_statistics(N_DF)
print(N_DF.head(1))

W_DF = apply_to_qa_df(
    N_DF,
    sentence_chunking_spacy,
    {},
    window_retrieval,
    {"window_size": 5, "retrieval_func": percentile_based_chunks, "retrieval_params": {"percentile": 15}},
    MODEL,
    doc_col="relevant_context",
    relevant_context_col="relevant_context_2",
)

W_DF = score_dataframe(W_DF, MODEL, col="relevant_context_2")
summary_statistics(W_DF, col="relevant_context_2")


Retrieving folder contents
Processing file 11Y7Njb06F7rJ_eHo-BWy-qqZn0NHFSST document-one-qa.txt
Processing file 1yJ-oOzV3qRuiAUd02q7wupNj9VncbYe_ document-one.txt
Processing file 1VKyrW2LLcVu7Aiai14Z_T_vWi8X5PDhf document-three-qa.txt
Processing file 1OwctcyBbFZeBUADVrGJphd1wzc1rRps1 document-three.txt
Processing file 1bNFl4jbL4oZq1nPySDU5_rjmY3X5trUM document-two-qa.txt
Processing file 12Ksdo1BSi7Yv4G1AgW7bnfgB6N0GgksK document-two.txt
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=11Y7Njb06F7rJ_eHo-BWy-qqZn0NHFSST
To: /content/examples/document-one-qa.txt
100% 1.19k/1.19k [00:00<00:00, 6.60MB/s]
Downloading...
From: https://drive.google.com/uc?id=1yJ-oOzV3qRuiAUd02q7wupNj9VncbYe_
To: /content/examples/document-one.txt
100% 11.8k/11.8k [00:00<00:00, 43.7MB/s]
Downloading...
From: https://drive.google.com/uc?id=1VKyrW2LLcVu7Aiai14Z_T_vWi8X5PDhf
To: /content/examples/document-t

<ipython-input-71-c2734cacb875>:911: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  avg_similarity = float(np.mean(rec_w_score_df['similarity_score']))


  Avg. Similarity: 0.4183
  Avg. Answer Presence: 0.2500
  Avg. Percentage: 25.9009
--------------------------------------------------
Running experiment for window_size=20 and percentile=90...


<ipython-input-71-c2734cacb875>:911: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  avg_similarity = float(np.mean(rec_w_score_df['similarity_score']))


  Avg. Similarity: 0.4183
  Avg. Answer Presence: 0.2500
  Avg. Percentage: 25.9009
--------------------------------------------------
Running experiment for window_size=20 and percentile=85...


...

  Avg. Similarity: 0.5088
  Avg. Answer Presence: 0.3333
  Avg. Percentage: 5.2731
--------------------------------------------------
Running experiment for window_size=1 and percentile=90...


<ipython-input-71-c2734cacb875>:911: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  avg_similarity = float(np.mean(rec_w_score_df['similarity_score']))


  Avg. Similarity: 0.5543
  Avg. Answer Presence: 0.5000
  Avg. Percentage: 10.8903
--------------------------------------------------
Running experiment for window_size=1 and percentile=85...


<ipython-input-71-c2734cacb875>:911: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  avg_similarity = float(np.mean(rec_w_score_df['similarity_score']))


  Avg. Similarity: 0.5147
  Avg. Answer Presence: 0.5000
  Avg. Percentage: 17.1418
--------------------------------------------------
Running experiment for window_size=1 and percentile=80...


In [3]:
import streamlit as st
import numpy as np
import pandas as pd
import re
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# -----------------------
# Helper functions
# -----------------------

def sentence_chunking_basic(text):
    """
    Split text into sentences based on punctuation.
    """
    sentences = re.split(r'(?<!\.\.\.)(?<=\.|\?|\!)(?=\s)', text.strip())
    return [s.strip() for s in sentences if s.strip()]

def embed_text(text, model):
    return model.encode(text)

def embed_chunks(chunks, model):
    return [embed_text(chunk, model) for chunk in chunks]

# -----------------------
# Streamlit app
# -----------------------

st.title("Document Chunking & Retrieval App")

st.markdown(
    """
    This application allows you to upload a text document, then it uses a simple sentence-chunking method to split the document into chunks.
    You can then enter a question, and the app will compute the cosine similarity between the question and each chunk (using a pre-trained SentenceTransformer model)
    and display the top 3 most similar chunks.
    """
)

# File uploader for text files
uploaded_file = st.file_uploader("Upload a text document (TXT file)", type=["txt"])

if uploaded_file is not None:
    # Read and decode the file content
    file_content = uploaded_file.read().decode("utf-8")

    st.header("Document Content")
    st.text(file_content)

    # Chunk the document using the basic sentence chunker
    chunks = sentence_chunking_basic(file_content)

    st.header("Document Chunks")
    for idx, chunk in enumerate(chunks):
        st.markdown(f"**Chunk {idx}:** {chunk}")

    # Enter a question
    question = st.text_input("Enter your question", "What is the rental price?")

    if question:
        st.header("Processing...")

        # Load the model (this may take a moment)
        with st.spinner("Loading model..."):
            model = SentenceTransformer('all-MiniLM-L6-v2')

        # Embed the question and document chunks
        question_embedding = embed_text(question, model).reshape(1, -1)
        chunk_embeddings = embed_chunks(chunks, model)
        # Ensure each embedding is reshaped to (1, -1)
        chunk_embeddings = [emb.reshape(1, -1) for emb in chunk_embeddings]

        # Compute cosine similarity between the question and each chunk
        similarities = [cosine_similarity(question_embedding, emb)[0][0] for emb in chunk_embeddings]

        # Display similarity scores (optional)
        st.subheader("Similarity Scores per Chunk")
        scores_df = pd.DataFrame({"Chunk Index": range(len(similarities)), "Similarity": similarities})
        st.dataframe(scores_df)

        # Retrieve top 3 chunks (you can adjust k as needed)
        k = 3
        top_indices = np.argsort(similarities)[::-1][:k]
        retrieved_chunks = [chunks[i] for i in top_indices]

        st.header("Retrieved Chunks")
        for i, chunk in enumerate(retrieved_chunks):
            st.markdown(f"**Chunk {i} (Index {top_indices[i]}):** {chunk}")


2025-03-15 12:04:46.434 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-03-15 12:04:46.689 
  command:

    streamlit run /usr/local/lib/python3.11/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2025-03-15 12:04:46.690 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-03-15 12:04:46.691 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-03-15 12:04:46.693 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-03-15 12:04:46.694 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-03-15 12:04:46.695 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-03-15 12:04:46.696 Thread 'MainThread': mi

In [4]:
! streamlit run /usr/local/lib/python3.11/dist-packages/colab_kernel_launcher.py [ARGUMENTS]





  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.126.183.95:8501

  Stopping...
^C


In [2]:
! pip install -q streamlit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 102.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 75.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 6.8 MB/s eta 0:00:00


In [ ]:
# from langchain.vectorstores import FAISS
# from langchain_community.vectorstores import FAISS
# import faiss
from itertools import combinations
from langchain.chains import RetrievalQA
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.llms import CTransformers
from langchain.schema import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_experimental.text_splitter import SemanticChunker
from langchain_text_splitters import RecursiveCharacterTextSplitter
from openai import OpenAI
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from typing import List, Tuple, Callable
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import re
import seaborn as sns


Thomas Decloedt, 12 March 2025

## Info

**Assumptions, requirements and choices:**

- Input documents won't have predictable structure
> Stick to chunking approach that does not assume a particular document structure of sections, paragraphs, etc.

- Exact extraction of text & prefer false positives over false negatives, i.e., over- rather than under-retrieval preferred
> No generation & put the exact occurence of answers in extracted text as priority when evaluating approaches

- For this MVP, I chose to only go with pre-trained sentence embedders, to keep it simple (no data engineering and training necessary)
and to illustrate problem solving ability
under constrained conditions

**Approach summary:**

I use two filtering steps:

1. An initial coarse one identifying the most relevant sentences and keeping the context centered around them.

2. A more fine-grained step to pinpoint the most important sentences among the previous extracted contexts.

Relevancy is computed using a pre-trained sentence embedder and the cosine similarity.


**Disclaimers:**

- ChatGPT used to generate functions for speed

- Certain parameters were set to try out the approach,
however, a modest dataset is necessary to set them such that the approach generalizes

## Installs

## Data

## Constants

In [ ]:
! pip install -q streamlit

In [5]:
import streamlit as st
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import re

# Load the pre-trained model once (cached for performance)
@st.cache_resource
def load_model():
    return SentenceTransformer('all-MiniLM-L6-v2')

MODEL = load_model()

# Basic sentence chunking function
def sentence_chunking_basic(text):
    sentences = re.split(r'(?<!\.\.\.)(?<=\.|\?|\!)(?=\s)', text.strip())
    return [sentence.strip() for sentence in sentences if sentence.strip()]

# Embedding functions
def embed_text(text, model):
    return model.encode(text)

def embed_chunks(chunks, model):
    return [embed_text(chunk, model) for chunk in chunks]

# Coarse filtering: Nth percentile with context window
def relevant_context(question, text, model, percentile=95, window_size=7):
    # Step 1: Chunk the text into sentences
    chunks = sentence_chunking_basic(text)

    # Step 2: Embed the question and chunks
    question_embedding = embed_text(question, model)
    chunk_embeddings = embed_chunks(chunks, model)

    # Step 3: Calculate cosine similarity
    similarities = [cosine_similarity([question_embedding], [chunk_embedding])[0][0]
                    for chunk_embedding in chunk_embeddings]

    # Step 4: Create DataFrame and mark top percentile
    df = pd.DataFrame({'Sentence': chunks, 'Similarity': similarities})
    threshold = df['Similarity'].quantile(percentile / 100)
    df['Top_Percentile'] = df['Similarity'] >= threshold

    # Step 5: Add context window around top percentile sentences
    df['In_Window'] = False
    top_indices = df[df['Top_Percentile']].index
    for idx in top_indices:
        start = max(0, idx - window_size // 2)
        end = min(len(df), idx + window_size // 2 + 1)
        df.loc[start:end, 'In_Window'] = True

    # Step 6: Concatenate sentences in the window
    return " ".join(df[df['In_Window']]['Sentence']), df

# Fine-grained filtering: Sliding window similarity
def window_retrieval(text, question, model, window_size=3, percentile=37.5):
    # Step 1: Chunk the coarse-filtered text
    chunks = sentence_chunking_basic(text)

    # Step 2: Embed question and chunks
    question_embedding = embed_text(question, model)
    chunk_embeddings = embed_chunks(chunks, model)

    # Step 3: Calculate similarities
    similarities = [cosine_similarity([question_embedding], [chunk_embedding])[0][0]
                    for chunk_embedding in chunk_embeddings]

    # Step 4: Update similarities with sliding window
    updated_similarities = []
    n = len(similarities)
    half_window = window_size // 2
    for i in range(n):
        start = max(0, i - half_window)
        end = min(n, i + half_window + 1)
        window = similarities[start:end]
        updated_similarities.append(max(window))

    # Step 5: Filter based on percentile
    df = pd.DataFrame({'Sentence': chunks, 'Similarity': updated_similarities})
    threshold = np.percentile(updated_similarities, percentile)
    relevant_chunks = df[df['Similarity'] >= threshold]['Sentence'].tolist()

    return " ".join(relevant_chunks), df

# Streamlit app layout
st.title("Document Question Extractor")
st.write("Enter a document and a question to extract relevant text using a two-step filtering approach.")

# Input fields
document = st.text_area("Paste your document here:", height=200)
question = st.text_input("Enter your question:")

# Parameters (optional customization)
col1, col2 = st.columns(2)
with col1:
    coarse_percentile = st.slider("Coarse Percentile", 50, 100, 95)
    coarse_window = st.slider("Coarse Window Size", 1, 15, 7)
with col2:
    fine_percentile = st.slider("Fine Percentile", 0, 50, 37)
    fine_window = st.slider("Fine Window Size", 1, 10, 3)

# Process button
if st.button("Extract Relevant Text"):
    if document and question:
        with st.spinner("Processing..."):
            # Step 1: Coarse filtering
            coarse_result, coarse_df = relevant_context(
                question, document, MODEL,
                percentile=coarse_percentile, window_size=coarse_window
            )

            # Step 2: Fine-grained filtering
            final_result, fine_df = window_retrieval(
                coarse_result, question, MODEL,
                window_size=fine_window, percentile=fine_percentile
            )

            # Display results
            st.subheader("Coarse Filtering Result")
            st.write(coarse_result)
            st.write(f"Length: {len(coarse_result)} characters ({len(coarse_result)/len(document)*100:.2f}% of original)")

            st.subheader("Final Extracted Text")
            st.write(final_result)
            st.write(f"Length: {len(final_result)} characters ({len(final_result)/len(document)*100:.2f}% of original)")

            # Optional: Show similarity scores for debugging
            with st.expander("View Similarity Scores"):
                st.write("Coarse Filtering Scores:")
                st.dataframe(coarse_df)
                st.write("Fine Filtering Scores:")
                st.dataframe(fine_df)
    else:
        st.error("Please provide both a document and a question.")

# Footer
st.write("Built with Streamlit and SentenceTransformers. Mimics a two-step extraction pipeline.")

2025-03-15 12:15:42.529 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-03-15 12:15:42.530 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-03-15 12:15:42.532 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-03-15 12:15:43.034 Thread 'Thread-8': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-03-15 12:15:43.035 Thread 'Thread-8': missing ScriptRunContext! This warning can be ignored when running in bare mode.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

2025-03-15 12:15:59.080 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-03-15 12:15:59.084 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-03-15 12:15:59.087 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-03-15 12:15:59.089 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-03-15 12:15:59.090 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-03-15 12:15:59.090 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-03-15 12:15:59.093 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-03-15 12:15:59.094 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [7]:
import streamlit as st
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import re

# Load the pre-trained model once (cached for performance)
@st.cache_resource
def load_model():
    return SentenceTransformer('all-MiniLM-L6-v2')

MODEL = load_model()

# Basic sentence chunking function
def sentence_chunking_basic(text):
    sentences = re.split(r'(?<!\.\.\.)(?<=\.|\?|\!)(?=\s)', text.strip())
    return [sentence.strip() for sentence in sentences if sentence.strip()]

# Embedding functions
def embed_text(text, model):
    return model.encode(text)

def embed_chunks(chunks, model):
    return [embed_text(chunk, model) for chunk in chunks]

# Coarse filtering: Nth percentile with context window
def relevant_context(question, text, model, percentile=95, window_size=7):
    # Step 1: Chunk the text into sentences
    chunks = sentence_chunking_basic(text)

    # Step 2: Embed the question and chunks
    question_embedding = embed_text(question, model)
    chunk_embeddings = embed_chunks(chunks, model)

    # Step 3: Calculate cosine similarity
    similarities = [cosine_similarity([question_embedding], [chunk_embedding])[0][0]
                    for chunk_embedding in chunk_embeddings]

    # Step 4: Create DataFrame and mark top percentile
    df = pd.DataFrame({'Sentence': chunks, 'Similarity': similarities})
    threshold = df['Similarity'].quantile(percentile / 100)
    df['Top_Percentile'] = df['Similarity'] >= threshold

    # Step 5: Add context window around top percentile sentences
    df['In_Window'] = False
    top_indices = df[df['Top_Percentile']].index
    for idx in top_indices:
        start = max(0, idx - window_size // 2)
        end = min(len(df), idx + window_size // 2 + 1)
        df.loc[start:end, 'In_Window'] = True

    # Step 6: Concatenate sentences in the window
    return " ".join(df[df['In_Window']]['Sentence']), df

# Fine-grained filtering: Sliding window similarity
def window_retrieval(text, question, model, window_size=3, percentile=37.5):
    # Step 1: Chunk the coarse-filtered text
    chunks = sentence_chunking_basic(text)

    # Step 2: Embed question and chunks
    question_embedding = embed_text(question, model)
    chunk_embeddings = embed_chunks(chunks, model)

    # Step 3: Calculate similarities
    similarities = [cosine_similarity([question_embedding], [chunk_embedding])[0][0]
                    for chunk_embedding in chunk_embeddings]

    # Step 4: Update similarities with sliding window
    updated_similarities = []
    n = len(similarities)
    half_window = window_size // 2
    for i in range(n):
        start = max(0, i - half_window)
        end = min(n, i + half_window + 1)
        window = similarities[start:end]
        updated_similarities.append(max(window))

    # Step 5: Filter based on percentile
    df = pd.DataFrame({'Sentence': chunks, 'Similarity': updated_similarities})
    threshold = np.percentile(updated_similarities, percentile)
    relevant_chunks = df[df['Similarity'] >= threshold]['Sentence'].tolist()

    return " ".join(relevant_chunks), df

# Streamlit app layout
st.title("Document Question Extractor")
st.write("Enter a document and a question to extract relevant text using a two-step filtering approach.")

# Input fields
document = st.text_area("Paste your document here:", height=200)
question = st.text_input("Enter your question:")

# Parameters (optional customization)
col1, col2 = st.columns(2)
with col1:
    coarse_percentile = st.slider("Coarse Percentile", 50, 100, 95)
    coarse_window = st.slider("Coarse Window Size", 1, 15, 7)
with col2:
    fine_percentile = st.slider("Fine Percentile", 0, 50, 37)
    fine_window = st.slider("Fine Window Size", 1, 10, 3)

# Process button
if st.button("Extract Relevant Text"):
    if document and question:
        with st.spinner("Processing..."):
            # Step 1: Coarse filtering
            coarse_result, coarse_df = relevant_context(
                question, document, MODEL,
                percentile=coarse_percentile, window_size=coarse_window
            )

            # Step 2: Fine-grained filtering
            final_result, fine_df = window_retrieval(
                coarse_result, question, MODEL,
                window_size=fine_window, percentile=fine_percentile
            )

            # Display results
            st.subheader("Coarse Filtering Result")
            st.write(coarse_result)
            st.write(f"Length: {len(coarse_result)} characters ({len(coarse_result)/len(document)*100:.2f}% of original)")

            st.subheader("Final Extracted Text")
            st.write(final_result)
            st.write(f"Length: {len(final_result)} characters ({len(final_result)/len(document)*100:.2f}% of original)")

            # Optional: Show similarity scores for debugging
            with st.expander("View Similarity Scores"):
                st.write("Coarse Filtering Scores:")
                st.dataframe(coarse_df)
                st.write("Fine Filtering Scores:")
                st.dataframe(fine_df)
    else:
        st.error("Please provide both a document and a question.")

# Footer
st.write("Built with Streamlit and SentenceTransformers. Mimics a two-step extraction pipeline.")


2025-03-15 12:20:02.909 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-03-15 12:20:02.917 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-03-15 12:20:02.918 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-03-15 12:20:02.920 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-03-15 12:20:02.922 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-03-15 12:20:02.923 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-03-15 12:20:02.924 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-03-15 12:20:02.925 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [6]:
! streamlit run app.py

Usage: streamlit run [OPTIONS] TARGET [ARGS]...
Try 'streamlit run --help' for help.

Error: Invalid value: File does not exist: app.py


In [ ]:
EXAMPLES_DIR = "/content/examples"

## Imports

# Data exploration

**Results**:

91.67% of answers obtained, while extracted content is 35% of the original source document on average.

(The one answer that was not retrieved was the extremely long one.)

As indicated by the exercise description,
further filtering with ChatGPT is of course possible.

**Note**:

The parameters were set manually by looking at performance on the three examples.
Thus these results serve merely as an indication,
not an actual evaluation.
To properly investigate the performance of the approach,
a dataset containing a train (for parameter tuning) and test (for evaluation) split is crucial.
I would imagine this approach could be trained with a modest dataset
containing tuples of (question, source document, extracted spans),
where the former two would be the input and the latter the output.
Training through grid search, for example,
optimizing the various parameters of the two steps, e.g., window size.
We could optimize using a custom scoring function that optimizes for recall and penalizes longer outputs.

$\text{Recall} = \frac{|\text{Relevant Items Retrieved}|}{|\text{Total Relevant Items in Ground Truth}|}$


**Approach summary:**

1. **Coarse Extraction**:
First, extract the **top nth percentile** of relevant text chunks (sentences), including a **context window** around them.
This reduces the search space while preserving useful contextual information.
Simple, fast, effective filtering.

2. **Fine-Grained Extraction**:
Apply a **sliding window with similarity scoring**,
smaller than the previous context window,
to the selected chunks to pinpoint the most relevant ones.
Somewhat more complex, but still simple, slower, practical further filtering.

**Conceptual Insight:**

- The combination of **sentence chunking, windowing, and similarity scoring** functions as a **simplified attention mechanism**, focusing on the most relevant parts of the text.

- This suggests the potential for a **transformer-based model** to learn **which chunks (better said, tokens) to extract**, assuming an appropriate dataset with **input questions, source documents, and expected extracted spans** is available.  

- In essence, **sentence chunking with sliding window similarity** mirrors the way attention mechanisms operate—albeit in a much simpler, heuristic-driven manner.  

See for example:
- [Extraction of Question-related Sentences for Reading Comprehension Tests via Attention Mechanism](https://ieeexplore.ieee.org/document/9382471)
- Span classification: [idea](https://stackoverflow.com/questions/70990722/which-model-technique-to-use-for-specific-sentence-extraction) and [source](https://huggingface.co/docs/transformers/model_doc/roberta#transformers.TFRobertaForQuestionAnswering)




# Other approaches, ideas & sources

I looked into multiple approaches, however, they were ultimately not chosen, for example, due to requiring too much engineering to work for this MVP.

- [Enhancing LLM Context with Recursive Summarization Using Python](https://github.com/xbeat/Machine-Learning/blob/main/Enhancing%20LLM%20Context%20with%20Recursive%20Summarization%20Using%20Python.md)

- [Chunking embedding fetching with OpenAI](https://gist.github.com/mburde7/73076b2e05b001a2779d812451b0b3ff)

- [Zchunk: A new chunking strategy developed by ZeroEntropy for general semantic chunking using Llama-70B](https://github.com/zeroentropy-ai/zchunk/tree/master)

- [LexNLP: a library for working with real, unstructured legal text, including contracts, plans, policies, procedures, and other material](https://lexpredict-lexnlp.readthedocs.io/en/latest/index.html)

- Chunk using [Legal NLP](https://johnsnowlabs.com)

- Chunk using HybridChunker form Docling

- Could add summaries or labels (pre-determined categories assigned with zero shot classifier) to chunks

- NLTKTextSplitter

- Coarse pre-processing of source documents, e.g., preamble & body categorization with a classifier
